In [2]:
"""
Build basis_master sheet: fc_rates
-----
Family Cover children rate schedule (2 age bands): 0-13 basic, 14+ basic + aids.
Source: Basis and Assumptions.xlsx > FC Basis - New > C3:E5.

FC row-4 formula wiring (V1.7.43):
  AW (kids 0-13 basic)   = 'FC Basis - New'!$D$4 * n_children_0_13
  AX (kids 14+ basic)    = 'FC Basis - New'!$D$5 * n_children_14_plus
  AY (kids 14+ aids)     = 'FC Basis - New'!$E$5 * n_children_14_plus

Author: Mohamed Hussein Gadiwala
Basis version: v2026_07 (basis master)
"""

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import csv
import openpyxl
import polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")

SOURCE_XLSX  = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "FC Basis - New"

BASIS_VERSION = "v2026_07"
SNAPSHOT_DIR = BASE / "basis_snapshots" / f"basis_{BASIS_VERSION}"
MASTER_XLSX  = BASE / "basis_master" / f"basis_master_{BASIS_VERSION}.xlsx"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

SHEET_NAME   = "fc_rates"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")
print(f"Source:   {SOURCE_XLSX.name} > {SOURCE_SHEET}")
print(f"Target:   {PARQUET_PATH}")
print(f"Loaded:   {ws.max_row} rows × {ws.max_column} cols")
print(f"Built at: {BUILT_AT}")


Building: fc_rates
Source:   Basis and Assumptions.xlsx > FC Basis - New
Target:   J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07\fc_rates.parquet
Loaded:   103 rows × 10 cols
Built at: 2026-07-22T10:44:44+00:00


In [3]:
def find_all_cells_any_col(ws, label: str, max_row: int = 1200, max_col: int = 30) -> list[tuple[int, int]]:
    """Find ALL cells matching `label` exactly, anywhere on the sheet."""
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits


def find_in_range(ws, label: str, row_start: int, row_end: int, col: int | None = None) -> tuple[int, int] | None:
    """Find first cell matching `label` within given row range (and optional col)."""
    hits = find_all_cells_any_col(ws, label)
    for r, c in hits:
        if row_start <= r < row_end and (col is None or c == col):
            return (r, c)
    return None


def get_col_letter(col: int) -> str:
    return openpyxl.utils.get_column_letter(col)


In [4]:
# Anchor: FC Basis - New!C3:E5 has header "START_AGE | BASIC_RATE | AIDS_RATE"
expected_headers = ["START_AGE", "BASIC_RATE", "AIDS_RATE"]
actual_headers = [ws.cell(row=3, column=c).value for c in (3, 4, 5)]
assert actual_headers == expected_headers, (
    f"FC Basis - New!C3:E3 header drift: got {actual_headers}, expected {expected_headers}"
)

all_rows: list[dict] = []
for excel_row in (4, 5):
    age_from   = ws.cell(row=excel_row, column=3).value
    basic_rate = ws.cell(row=excel_row, column=4).value
    aids_rate  = ws.cell(row=excel_row, column=5).value

    assert isinstance(age_from, (int, float))
    assert isinstance(basic_rate, (int, float))
    assert isinstance(aids_rate, (int, float))

    all_rows.append({
        "product":       "fc",
        "age_from":      int(age_from),
        "basic_rate":    float(basic_rate),
        "aids_rate":     float(aids_rate),
        "source_label":  f"FC Basis - New!D{excel_row}, E{excel_row}",
        "source_row":    excel_row,
        "notes": ("Children basic + aids rates per R1000 cover. "
                  "Applied by FC pricing via n_children_0_13 and n_children_14_plus."),
    })

print(f"Total rows harvested: {len(all_rows)}")
for r in all_rows:
    print(f"  age_from={r['age_from']:>3d}  basic={r['basic_rate']:.6f}  "
          f"aids={r['aids_rate']:.6f}  src=row{r['source_row']}")


Total rows harvested: 2
  age_from=  0  basic=1.119133  aids=0.000000  src=row4
  age_from= 14  basic=0.651755  aids=0.378190  src=row5


In [5]:
df = pl.DataFrame(all_rows).select([
    "product", "age_from", "basic_rate", "aids_rate",
    "source_label", "source_row", "notes",
])

key_cols = ["product", "age_from"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

expected_products = {"fc"}
actual_products = set(df["product"].unique().to_list())
assert actual_products == expected_products

def get_rate(age_from, col):
    row = df.filter(pl.col("age_from") == age_from)
    assert row.height == 1
    return float(row[col][0])

assert abs(get_rate(0, "basic_rate")  - 1.11913253117143) < 1e-9
assert abs(get_rate(0, "aids_rate")   - 0.0) < 1e-9
assert abs(get_rate(14, "basic_rate") - 0.651755343823977) < 1e-9
assert abs(get_rate(14, "aids_rate")  - 0.378189613982071) < 1e-9

print(f"✓ {df.height} rows validated")
print(f"✓ 2 age bands present (0-13, 14+)")
print(f"✓ 4 frozen numeric checks passed")
print(f"\nFinal DataFrame:")
print(df)


✓ 2 rows validated
✓ 2 age bands present (0-13, 14+)
✓ 4 frozen numeric checks passed

Final DataFrame:
shape: (2, 7)
┌─────────┬──────────┬────────────┬───────────┬──────────────┬────────────┬────────────────────────┐
│ product ┆ age_from ┆ basic_rate ┆ aids_rate ┆ source_label ┆ source_row ┆ notes                  │
│ ---     ┆ ---      ┆ ---        ┆ ---       ┆ ---          ┆ ---        ┆ ---                    │
│ str     ┆ i64      ┆ f64        ┆ f64       ┆ str          ┆ i64        ┆ str                    │
╞═════════╪══════════╪════════════╪═══════════╪══════════════╪════════════╪════════════════════════╡
│ fc      ┆ 0        ┆ 1.119133   ┆ 0.0       ┆ FC Basis -   ┆ 4          ┆ Children basic + aids  │
│         ┆          ┆            ┆           ┆ New!D4, E4   ┆            ┆ rates pe…              │
│ fc      ┆ 14       ┆ 0.651755   ┆ 0.37819   ┆ FC Basis -   ┆ 5          ┆ Children basic + aids  │
│         ┆          ┆            ┆           ┆ New!D5, E5   ┆            

In [6]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

if MASTER_XLSX.exists():
    master_wb = openpyxl.load_workbook(MASTER_XLSX)
else:
    master_wb = openpyxl.Workbook()
    if "Sheet" in master_wb.sheetnames:
        del master_wb["Sheet"]

if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")


✓ Wrote fc_rates.parquet
  Size:    3.17 KB
  SHA-256: deca50b6e6265cb0...
✓ Master workbook updated: basis_master_v2026_07.xlsx


In [7]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "basis_version", "notes",
]

log_row = {
    "built_at": BUILT_AT, "built_by": BUILT_BY, "sheet": SHEET_NAME,
    "rows": df.height, "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}", "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name, "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA, "basis_version": BASIS_VERSION,
    "notes": ("FC children rate schedule. 2 age bands (0-13, 14+). "
              "Anchored on FC Basis - New!C3:E5. Per R1000 cover. "
              "Consumer: FC row-4 formulas AW/AX/AY."),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated ({BUILD_LOG.name})")
print(f"\n{'='*60}\nBUILD COMPLETE: {SHEET_NAME}\n{'='*60}")


✓ build_log.csv updated (build_log.csv)

BUILD COMPLETE: fc_rates
